In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
        "t2m",
        "tp",
        "slhf",
        "sshf",
        "ssrd",
        "fal",
        "str",
    ]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"]

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'CEU']
experiment_region_groups = {"CEU": ["FR", "IT_AT", "CH"]}
res_xreg_by_source = {}
split_info_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Source region: {src_region}")

    res_xreg, split_info = prepare_monthly_df_xreg_SOURCE_to_EU(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        run_flag=True,
        source_code=src_region,
        region_groups=experiment_region_groups)

    res_xreg_by_source[src_region] = res_xreg
    split_info_by_source[src_region] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    print(f"Train glaciers ({src_region}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers (non-{src_region}):",
          len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "Test rows:", len(df_test))

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}  # set to empty set to include all

res_all_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH":  ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "CEU": ["FR", "IT_AT", "CH"],
}

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    ceu_members_excl_src = [
        c for c in ["FR", "IT_AT", "CH"] if c not in src_members
    ]
    experiment_region_groups = {}
    if ceu_members_excl_src:
        experiment_region_groups["CEU"] = ceu_members_excl_src

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=None,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups=experiment_region_groups,
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

In [ ]:
RECOMPUTE_SHIFTS = True

CLIMATE_COLS = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE']
TOPO_COLS = ['aspect', 'slope', 'svf']

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache_basic.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s = build_global_scalers_multi_source(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=CLIMATE_COLS,
                static_cols=TOPO_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                compute_marginals=False,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

In [ ]:
# # Plot per-pair shifts
# for src_region, all_shifts in all_shifts_by_source.items():
#     for key, shift in all_shifts.items():
#         tgt = key.split('_TO_')[1]
#         print(f"Shift for {key}:")
#         fig = plot_domain_shift(shift,
#                                 CLIMATE_COLS,
#                                 TOPO_COLS,
#                                 src=src_region,
#                                 tgt=tgt)
#         plt.show()

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

#### KDEs:

In [ ]:
# scaler_m_pmb, scaler_s_pmb = build_global_scalers_multi_source(
#     res_xreg_by_source,
#     monthly_cols=MONTHLY_COLS + ["POINT_BALANCE"],
#     static_cols=STATIC_COLS,
# )

# # for src_region in SOURCE_REGIONS:
# #     figs = plot_feature_kde_overlap_xreg_with_shifts(
# #         res_all_xreg=res_all_xreg_by_source[src_region],
# #         cfg=cfg,
# #         monthly_cols=MONTHLY_COLS,
# #         static_cols=STATIC_COLS,
# #         scaler_m=scaler_m_pmb,
# #         scaler_s=scaler_s_pmb,
# #         output_dir=f"figures/xreg_kde_with_shifts/{src_region}",
# #     )

# figs = plot_feature_kde_overlap_xreg_with_shifts(
#     res_all_xreg=res_all_xreg_by_source["CH"],
#     cfg=cfg,
#     monthly_cols=MONTHLY_COLS,
#     static_cols=STATIC_COLS,
#     scaler_m=scaler_m_pmb,
#     scaler_s=scaler_s_pmb,
#     output_dir=f"figures/xreg_kde_with_shifts/{src_region}",
# )

#### T-sne:

In [ ]:
# for src_region in SOURCE_REGIONS:
#     figs_by_code = plot_tsne_overlap_xreg_from_single_res(
#         res_xreg=res_xreg_by_source[src_region],
#         cfg=cfg,
#         STATIC_COLS=STATIC_COLS,
#         MONTHLY_COLS=MONTHLY_COLS,
#         group_col="SOURCE_CODE",
#         target_code=src_region,
#         use_aug=False,
#         n_iter=1000,
#     )

## LSTM model
### LSTM datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "LSTM_cache"
outputs_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH": ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "CEU": ["FR", "IT_AT", "CH"],
}

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    ceu_members_excl_src = [
        c for c in ["FR", "IT_AT", "CH"] if c not in src_members
    ]
    experiment_region_groups = {}
    if ceu_members_excl_src:
        experiment_region_groups["CEU"] = ceu_members_excl_src

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        region_groups=experiment_region_groups,
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

### LSTM parameters:

In [ ]:
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

### Train model:

In [ ]:
def train_or_load_one_source_model(
    cfg,
    key: str,
    lstm_assets: dict,  # only needs ds_train, train_idx, val_idx
    best_params: dict,
    device,
    models_dir="models",
    prefix="lstm_xreg",
    train_flag=True,
    force_retrain=False,
    epochs=150,
    batch_size_train=64,
    batch_size_val=128,
    verbose=True,
    date=None,
):
    """Train or load a single source-region model. No test set needed."""
    run_date = datetime.now().strftime("%Y-%m-%d") if date is None else date
    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(models_dir, f"{prefix}_{key}_{run_date}.pt")

    model = mbm.models.LSTM_MB.build_model_from_params(cfg,
                                                       best_params,
                                                       device,
                                                       verbose=verbose)
    loss_fn = mbm.models.LSTM_MB.resolve_loss_fn(best_params)

    # Load existing checkpoint if available and not forcing retrain
    if train_flag and (not force_retrain) and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    if not train_flag and not os.path.exists(model_filename):
        raise FileNotFoundError(
            f"No checkpoint found for {key}: {model_filename}")

    if not train_flag and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    # --- Train ---
    mbm.utils.seed_all(cfg.seed)

    ds_train = lstm_assets["ds_train"]
    train_idx = lstm_assets["train_idx"]
    val_idx = lstm_assets["val_idx"]

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_train)

    train_dl, val_dl = ds_train_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        batch_size_train=batch_size_train,
        batch_size_val=batch_size_val,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=verbose,
    )

    if os.path.exists(model_filename):
        os.remove(model_filename)
        if verbose:
            print(f"Deleted existing model file: {model_filename}")

    history, best_val, best_state = model.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        clip_val=1,
        sched_factor=0.5,
        sched_patience=6,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-6,
        es_patience=15,
        es_min_delta=1e-4,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
        loss_fn=loss_fn,
    )

    if verbose:
        plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model.load_state_dict(state)

    return model, model_filename, {"history": history, "best_val": best_val}


def prepare_test_loader_for_target(
    cfg,
    model,
    lstm_assets_src: dict,  # needs ds_train (for scaler fitting)
    lstm_assets_tgt: dict,  # needs ds_test
    batch_size_test=128,
):
    """Given a trained model and a target's test assets, build the test loader."""
    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_src["ds_train"])
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_tgt["ds_test"])

    # fit scaler on train, apply to test
    ds_train_copy.make_loaders(
        train_idx=lstm_assets_src["train_idx"],
        val_idx=lstm_assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
    )

    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_train_copy, batch_size=batch_size_test, seed=cfg.seed)

    return test_dl, ds_test_copy

In [ ]:
models_by_source = {}
infos_by_source = {}

TRAIN_FLAG = True  # set to True to retrain from scratch
MODEL_DATE = "2026-04-13"  # pin date so filenames are stable across runs

for src_region in SOURCE_REGIONS:
    # Pick any key to get the train assets — they're the same across all targets
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=default_params,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_xreg_{src_region}",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        date=MODEL_DATE,
    )

    models_by_source[src_region] = model
    infos_by_source[src_region] = {"model_path": model_path, **(info or {})}
    print(f"{src_region} -> model trained/loaded from {model_path}")

### Evaluate on test:

In [ ]:
df_metrics_by_source = {}
preds_by_source = {}
figs_by_source = {}

# Preferred display order for target regions — any not listed appear at the end
DISPLAY_ORDER = ["CEU", "FR", "NOR", "ISL", "SJM", "ALA", "CAW"]

for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = list(outputs_xreg_by_source[src_region].keys())
    models_by_key = {key: models_by_source[src_region] for key in target_keys}

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(2, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source[src_region] = df_metrics
    preds_by_source[src_region] = preds_by_key
    figs_by_source[src_region] = figs_by_key

### Region shift vs performance:

#### Distance topo & climate & elv both:

In [ ]:
EXCLUDE_TARGETS = ['CAW'] 

# Combine across all sources
df_metrics_all = pd.concat(df_metrics_by_source.values())
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source[src_region])

# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual", "RMSE_winter"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
)

In [ ]:
RECOMPUTE_SHIFTS = True

CLIMATE_COLS = ['t2m', 'tp', 'ssrd', 'ELEVATION_DIFFERENCE', 'str', 'fal']
TOPO_COLS = ['aspect', 'slope', 'svf']

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache_ebm.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s = build_global_scalers_multi_source(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=CLIMATE_COLS,
                static_cols=TOPO_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                compute_marginals=False,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

In [ ]:
# EXCLUDE_TARGETS = ["USCA"]  # set to [] to include all
EXCLUDE_TARGETS = ['CAW']

# Combine across all sources
df_metrics_all = pd.concat(df_metrics_by_source.values())
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source[src_region])

# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual", "RMSE_winter"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
)